# Parliamentary AI Assistant: Retrieval-Augmented Generation (RAG)

## 1. Project Overview

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using parliamentary proceedings from the 17th Lok Sabha (2019–2024).

The system combines semantic search with a locally hosted Large Language Model (LLM) to answer questions using retrieved passages from parliamentary documents.

Unlike a standalone language model that generates answers from its pretrained knowledge, a RAG system retrieves relevant source material at query time and supplies it as context for answer generation.

### Objectives

- Process parliamentary documents into searchable text chunks.
- Generate vector embeddings for semantic retrieval.
- Retrieve relevant passages using a FAISS vector index.
- Generate context-grounded answers using Llama through Ollama.
- Display source PDF page references to support verification.
- Establish a reproducible foundation for integrating the pipeline into a Streamlit application.

### Technology Stack

| Component | Technology | Purpose |
|---|---|---|
| Data processing | Python | Load and prepare parliamentary text |
| Document splitting | LangChain Text Splitters | Divide text into manageable chunks |
| Embeddings | Sentence Transformers | Represent text as numerical vectors |
| Vector search | FAISS | Retrieve semantically relevant passages |
| Language model | Llama 3.2 (3B), via Ollama | Generate answers from retrieved context |
| Orchestration | LangChain | Connect retrieval, prompts, and generation |

### Scope

The current prototype uses the English version of the President's Address delivered on 31 January 2024, extracted from the parliamentary proceedings PDF. It is a focused proof of concept, not yet a comprehensive assistant covering all parliamentary debates, questions, and speeches.

**Expected outcome:** a working question-answering pipeline that retrieves evidence from the selected document, generates an answer, and reports the PDF pages consulted.

## 2. RAG Architecture

The pipeline follows three main stages: retrieval, context augmentation, and answer generation.

1. **Document preparation:** Load cleaned parliamentary records and split them into overlapping text chunks.
2. **Embedding and indexing:** Convert each chunk into a vector representation and store the vectors in a FAISS index.
3. **Retrieval:** Convert the user's question into an embedding and retrieve the most relevant chunks.
4. **Context augmentation:** Insert the retrieved passages and their source-page metadata into a prompt.
5. **Generation:** Use Llama to formulate an answer from the supplied evidence.
6. **Source reporting:** Display the PDF pages associated with the retrieved passages so the answer can be checked against the source.

### Pipeline flow

`Parliamentary PDF → Extracted text → Chunks → Embeddings → FAISS`

`User question → FAISS retrieval → Retrieved context → Llama → Answer + source pages`

### Important limitation

Displaying retrieved source pages does not automatically prove that every statement in the generated answer is supported by those pages. Citation membership and factual grounding must be evaluated separately.

In [1]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama

# Project paths
PROJECT_DIR = Path.cwd()

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "president_address_en_cleaned.jsonl"
)

INDEX_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "faiss_president_address"
)

# Model configuration
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "llama3.2:3b"

print("Project directory:", PROJECT_DIR)
print("Cleaned data exists:", DATA_PATH.exists())
print("Saved FAISS index exists:", (INDEX_PATH / "index.faiss").exists())

C:\Users\Tony\AppData\Local\Temp\ipykernel_14320\2603105506.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Project directory: C:\Users\Tony\Documents\Lok_Sabha_AI
Cleaned data exists: True
Saved FAISS index exists: True


## 3. Load the Cleaned Parliamentary Records

The cleaned JSONL file contains the extracted English text from the President's Address to Parliament on 31 January 2024.

Each record preserves the source document, PDF page number, language, extraction method, and extracted text.

We load these records to prepare the source material for chunking and retrieval.

In [4]:
# Loading the cleaned parliamentary records
with DATA_PATH.open("r", encoding="utf-8") as file:
    records = [json.loads(line) for line in file if line.strip()]

print("Records loaded:", len(records))
print("Unique PDF pages:", len({record["pdf_page"] for record in records}))
print("\nFirst record metadata:")

{
    key: value
    for key, value in records[0].items()
    if key != "text"
}

Records loaded: 27
Unique PDF pages: 27

First record metadata:


{'document_id': 'president_address_2024_01_31',
 'source_file': 'Floor.pdf',
 'pdf_page': 60,
 'language': 'en',
 'extraction_method': 'direct'}

## 4. Split Documents into Chunks

Large documents are difficult to process and retrieve as a single unit. We divide the extracted text into smaller, overlapping chunks.

- **Chunk size:** 500 characters
- **Chunk overlap:** 50 characters
- **Purpose:** Preserve useful context across chunk boundaries while enabling more targeted retrieval.

Each chunk retains its original metadata, including its PDF page number, so retrieved passages can be traced back to the source.

In [5]:
# Split the parliamentary records into overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

documents = [
    Document(
        page_content=record["text"],
        metadata={
            key: value
            for key, value in record.items()
            if key != "text"
        }
    )
    for record in records
]

chunks = text_splitter.split_documents(documents)

print("Source documents:", len(documents))
print("Total chunks:", len(chunks))
print("First chunk source page:", chunks[0].metadata["pdf_page"])
print("First chunk characters:", len(chunks[0].page_content))

Source documents: 27
Total chunks: 103
First chunk source page: 60
First chunk characters: 439


## 5. Generate Text Embeddings

Text embeddings represent passages as numerical vectors that capture aspects of their semantic meaning.

We use the pretrained `sentence-transformers/all-MiniLM-L6-v2` model to embed each chunk. These vectors allow the retrieval system to identify passages that are semantically related to a user's question, even when the wording differs.

The same embedding model must be used when building and querying the vector index.

In [6]:
# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding model initialized successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding model initialized successfully.


## 6. Loading the FAISS Vector Index

FAISS (Facebook AI Similarity Search) enables efficient similarity-based retrieval over the stored text embeddings.

We load the previously saved index instead of rebuilding it on every notebook run. The index contains the embedded chunks and their associated document data, including source metadata.

The index is loaded using the same embedding model that was used to create it.

In [8]:
# Load the existing FAISS index
vector_store = FAISS.load_local(
    str(INDEX_PATH),
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded successfully.")
print("Indexed chunks:", vector_store.index.ntotal)

FAISS index loaded successfully.
Indexed chunks: 103


## 7. Initialize the Local Language Model

We use `llama3.2:3b`, a locally running language model accessed through Ollama, to generate answers from retrieved parliamentary passages.

Running the model locally allows us to experiment with question answering without relying on a hosted language-model API.

The model is configured with a temperature of zero to encourage consistent, less-random responses. This setting does not guarantee factual accuracy or prevent hallucinations.

In [9]:
# Initialize the local language model
llm = ChatOllama(
    model=LLM_MODEL_NAME,
    temperature=0
)

print("Language model initialized:", LLM_MODEL_NAME)

Language model initialized: llama3.2:3b


## 8. Build the RAG Question-Answering Pipeline

The Retrieval-Augmented Generation (RAG) pipeline combines semantic search with a language model.

For each question, the pipeline:

1. Retrieves the three most relevant chunks from the FAISS index.
2. Combines the retrieved passages into a context.
3. Sends the context and question to the local language model.
4. Generates an answer instructed to rely on the supplied passages.
5. Displays the PDF pages associated with the retrieved chunks.

The displayed source pages identify the material retrieved by the system. They do not independently

In [10]:
# Define the prompt used to generate answers from retrieved passages
rag_prompt = ChatPromptTemplate.from_template("""
You are a parliamentary research assistant.

Answer using only the supplied source passages.
Do not invent facts or add unsupported claims.
If the evidence is insufficient, say so.
Do not write page citations in the answer.
The application will list the retrieved source pages separately.

SOURCE PASSAGES:
{context}

QUESTION:
{question}

ANSWER:
""")

# Retrieve relevant passages and generate an answer
def ask_parliament(question, k=3):
    docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[PDF page {doc.metadata['pdf_page']}]\n{doc.page_content}"
        for doc in docs
    )

    response = llm.invoke(
        rag_prompt.format_messages(
            context=context,
            question=question
        )
    )

    source_pages = sorted({
        int(doc.metadata["pdf_page"])
        for doc in docs
    })

    print("Question:", question)
    print("\nAnswer:\n", response.content.strip())
    print(
        "\nRetrieved source pages:",
        ", ".join(f"PDF p. {page}" for page in source_pages)
    )

    return response.content.strip(), docs

print("RAG question-answering pipeline is ready.")

RAG question-answering pipeline is ready.


## 9. Test the RAG Pipeline

We evaluate the pipeline using a question about India's space programme mentioned in the President's Address.

This test checks whether the system can retrieve relevant passages, generate an answer from the supplied context, and display the associated PDF page numbers.

A successful execution confirms that the components work together. Answer accuracy and retrieval relevance must still be assessed separately.

In [11]:
# Testing the pipeline with a question about India's space programme
answer, retrieved_docs = ask_parliament(
    "What did the President say about India's space programme?"
)

Question: What did the President say about India's space programme?

Answer:
 The President mentioned that major decisions have been taken to expand India's space programme, which has led to the formation of many new space startups. They also mentioned that India's Gaganyaan will reach space, and that the government is engaged in making India a major space power in the world, which is a means to improve human life and increase India's share in the space economy.

Retrieved source pages: PDF p. 66, PDF p. 80, PDF p. 84


## 10. Initial Test Results

The pipeline successfully generated an answer to a question about India's space programme using the indexed English text of the President's Address.

**Observed results**
- The question was processed successfully.
- The model generated a natural-language response.
- Three retrieved source pages were displayed.
- The response discussed programme expansion, space startups, and Gaganyaan.

**Limitations**
- Retrieved pages may vary in relevance.
- Displaying source pages does not prove that every answer claim is supported.
- The model may omit details or introduce unsupported statements.
- This prototype covers only the indexed English President's Address, not all parliamentary proceedings.

Further work should evaluate retrieval relevance, answer grounding, and citation accuracy using multiple test questions.

## 11. Current Project Status

The initial local RAG pipeline is operational.

**Completed**
- Loaded cleaned parliamentary records.
- Split source text into overlapping chunks.
- Initialized a pretrained embedding model.
- Loaded the FAISS vector index.
- Initialized the local Llama 3.2 language model.
- Connected retrieval and answer generation.
- Tested the pipeline with a parliamentary question.

**Next steps**
1. Test additional questions, including questions with insufficient evidence.
2. Inspect retrieved passages and improve retrieval relevance.
3. Evaluate answer grounding systematically.
4. Prepare the pipeline for integration into a Streamlit application.